In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch08-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "08-01-chemical-space.py"

# 用兩種尺度找鄰居

第一次執行需下載 ESOL。小練習可接在寫入分析程式的儲存格末端，重跑該格與末格即可。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import deepchem as dc
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import Crippen, Descriptors, rdFingerprintGenerator
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

tasks, (dataset,), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None, transformers=[],
    reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
records = []
for s, y, w in zip(dataset.ids, dataset.y[:, 0], dataset.w[:, 0]):
    mol = Chem.MolFromSmiles(s)
    if mol is not None and w > 0 and np.isfinite(y):
        records.append((Chem.MolToSmiles(mol), Descriptors.MolWt(mol),
                        Crippen.MolLogP(mol), float(y)))
table = (pd.DataFrame(records, columns=["structure", "MW", "cLogP", "logS"])
         .groupby("structure", sort=True).median())
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [generator.GetFingerprint(Chem.MolFromSmiles(s)) for s in table.index]
query = Chem.MolToSmiles(Chem.MolFromSmiles("Cc1ccc(O)cc1"))  # 對甲酚
query_fp = fps[table.index.get_loc(query)]
table["Tc"] = DataStructs.BulkTanimotoSimilarity(query_fp, fps)
axes_cols = ["MW", "cLogP"]
offset = (table[axes_cols] - table.loc[query, axes_cols]) / table[axes_cols].std(ddof=0)
table["distance"] = np.sqrt((offset ** 2).sum(axis=1))
others = table.drop(index=query)
print("records / unique structures:", len(records), len(table))
print("query:", query, table.loc[query].round(4).to_dict())
print("Closest in descriptor space:")
print(others.sort_values("distance", kind="stable").head(5).round(5).to_string())
print("Closest by fingerprint:")
print(others.sort_values("Tc", ascending=False, kind="stable").head(5).round(5).to_string())

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4.4), layout="constrained")
dots = left.scatter(table.MW, table.cLogP, c=table.logS, s=12, alpha=.75, cmap="viridis")
left.plot(table.at[query,"MW"], table.at[query,"cLogP"], marker="*", markersize=13,
          color="crimson", label="p-Cresol")
left.set(xlabel="Molar mass (g/mol)", ylabel="Calculated logP")
left.legend()
fig.colorbar(dots, ax=left, label="Measured log S (mol/L, log10 scale)")
close = others.sort_values("distance", kind="stable").head(5)
right.barh(range(len(close)), close.Tc, color="#307b72")
right.set_yticks(range(len(close)), labels=close.index)
right.invert_yaxis()
right.set(xlim=(0, 1), xlabel="Morgan Tanimoto to p-cresol", title="Five nearest by MW and cLogP")
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
fig.savefig(out/"08-01-chemical-space.png", dpi=165)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "08-01-chemical-space.png")))